Maximal Marginal Relevance

In [1]:
# libraries
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

Loading and chunking the document

In [2]:
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

Building Retriever via FIASS Vectorstore and HuggingFace Embeddings

In [3]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks,embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

creating a MMR retriever

In [4]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3}
)

building rag chain w mmr retriever

prompt template

In [5]:
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.
        
Context:
{context}
                
Question: {input}
""")

initializing LLM

In [6]:
llm = init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022521C982F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022521C99D30>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

building the document chain

In [7]:
document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context provided.\n\nContext:\n{context}\n\nQuestion: {input}\n')
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022521C982F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022521C99D30>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)
| StrOutputParser(), kwa

final rag chain/pipeline

In [8]:
rag_chain = create_retrieval_chain(retriever,combine_docs_chain=document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000022521AD57F0>, search_type='mmr', search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context provided.\n\nContext:\n{context}\n\nQuestion: {input}\n')
            | ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_i

testing out wrt query

In [9]:
query = {"input":"How does Langchain support agents and memory?"}
response = rag_chain.invoke(query)
print("Answer:\n",response["answer"])

Answer:
 LangChain supports agents and memory in the following ways:

1. **Agent Capabilities**: LangChain agents can use various tools such as calculators, search APIs, or custom functions based on the instructions they receive. This allows agents to perform a wide range of tasks and interact with external data sources.

2. **Conversational Memory**: LangChain provides two types of memory support:
   - **ConversationBufferMemory**: This allows models to retain previous interactions, making multi-turn conversations more coherent.
   - **ConversationSummaryMemory**: This enables summarization memory, where agents can summarize previous conversations and use that information to inform their responses.

3. **Agent Decision Making**: LangChain allows LLMs to act as agents that decide which tool to call and in what order during a task. This enables agents to dynamically choose the most suitable tools to complete a task, enhancing their capabilities and flexibility.

4. **External Data Inter

relevant docs returned by MMR Retriever

In [10]:
response

{'input': 'How does Langchain support agents and memory?',
 'context': [Document(id='1dcb9f98-e613-4b5a-8172-2c12d753cb2b', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='9fd88690-502f-4ac2-9322-43742da512f5', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.'),
  Document(id='c18665a6-8585-4d4b-adb2-d8965d59e448', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order duri